In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))
print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.6 GB


In [2]:
import os

dirs = [
    '/kaggle/working/models/enhancement',
    '/kaggle/working/models/superresolution',
    '/kaggle/working/models/knowledge_base',
    '/kaggle/working/pipeline',
    '/kaggle/working/training',
    '/kaggle/working/data/degraded',
    '/kaggle/working/data/clean',
    '/kaggle/working/weights',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    open(f'{d}/__init__.py', 'w').close()

open('/kaggle/working/models/__init__.py', 'w').close()
print("✅ All folders created")


✅ All folders created


In [3]:
import os
import shutil
from pathlib import Path

PAIRED_SOURCES = [
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/trainA",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/trainB",
        "tag": "imagenet_train"
    },
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_dark/trainA",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_dark/trainB",
        "tag": "dark_train"
    },
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_scenes/trainA",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_scenes/trainB",
        "tag": "scenes_train"
    },
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/validation",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/validation",
        "tag": "imagenet_val"
    },
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_dark/validation",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_dark/validation",
        "tag": "dark_val"
    },
    {
        "degraded": "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/test_samples/Inp",
        "clean":    "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/test_samples/GTr",
        "tag": "test_samples"
    },
]

OUT_DEG = '/kaggle/working/data/degraded'
OUT_CLN = '/kaggle/working/data/clean'
os.makedirs(OUT_DEG, exist_ok=True)
os.makedirs(OUT_CLN, exist_ok=True)
exts = ['.jpg', '.jpeg', '.png']
total = 0

for src in PAIRED_SOURCES:
    deg_files = sorted([f for f in os.listdir(src["degraded"])
                       if Path(f).suffix.lower() in exts])

    if src["degraded"] == src["clean"]:
        for fname in deg_files:
            new_name = f"{src['tag']}_{fname}"
            shutil.copy2(
                os.path.join(src["degraded"], fname),
                os.path.join(OUT_DEG, new_name)
            )
            shutil.copy2(
                os.path.join(src["clean"], fname),
                os.path.join(OUT_CLN, new_name)
            )
            total += 1
    else:
        for fname in deg_files:
            clean_src = os.path.join(src["clean"], fname)
            if os.path.exists(clean_src):
                new_name = f"{src['tag']}_{fname}"
                shutil.copy2(
                    os.path.join(src["degraded"], fname),
                    os.path.join(OUT_DEG, new_name)
                )
                shutil.copy2(clean_src, os.path.join(OUT_CLN, new_name))
                total += 1

print(f"✅ Total paired images: {total}")
print(f"   Degraded: {len(os.listdir(OUT_DEG))}")
print(f"   Clean:    {len(os.listdir(OUT_CLN))}")


✅ Total paired images: 13790
   Degraded: 13791
   Clean:    13791


In [4]:
generator_code = r'''
import torch
import torch.nn as nn

torch.manual_seed(42)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False) if down
            else nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch) if use_bn else nn.Identity(),
            nn.LeakyReLU(0.2) if down else nn.ReLU(),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=64):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, features, down=True, use_bn=False)
        self.enc2 = ConvBlock(features, features*2, down=True)
        self.enc3 = ConvBlock(features*2, features*4, down=True)
        self.enc4 = ConvBlock(features*4, features*8, down=True)
        self.attention = ChannelAttention(features*8)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features*8, features*8, 4, 2, 1), nn.ReLU()
        )
        self.dec1 = ConvBlock(features*8,   features*8, down=False, dropout=True)
        self.dec2 = ConvBlock(features*8*2, features*4, down=False, dropout=True)
        self.dec3 = ConvBlock(features*4*2, features*2, down=False)
        self.dec4 = ConvBlock(features*2*2, features,   down=False)
        self.output = nn.Sequential(
            nn.ConvTranspose2d(features*2, out_channels, 4, 2, 1), nn.Tanh()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.attention(self.enc4(e3))
        b  = self.bottleneck(e4)
        d1 = self.dec1(b)
        d2 = self.dec2(torch.cat([d1, e4], dim=1))
        d3 = self.dec3(torch.cat([d2, e3], dim=1))
        d4 = self.dec4(torch.cat([d3, e2], dim=1))
        return self.output(torch.cat([d4, e1], dim=1))
'''
with open('/kaggle/working/models/enhancement/generator.py', 'w') as f:
    f.write(generator_code)
print("✅ Generator written")


✅ Generator written


In [5]:
loss_code = r'''
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import VGG16_Weights

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=VGG16_Weights.DEFAULT)
        self.feat = nn.Sequential(*list(vgg.features)[:16])
        for p in self.feat.parameters():
            p.requires_grad = False
        self.l1 = nn.L1Loss()
    def forward(self, x, y):
        return self.l1(self.feat(x), self.feat(y))

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.ws = window_size
        self.ch = 3
        import math
        sigma = 1.5
        g = torch.Tensor([math.exp(-(x-window_size//2)**2/(2*sigma**2)) for x in range(window_size)])
        g = g / g.sum()
        w = g.unsqueeze(1).mm(g.unsqueeze(0)).float().unsqueeze(0).unsqueeze(0)
        self.register_buffer('window', w.expand(3,1,window_size,window_size).contiguous())

    def forward(self, img1, img2):
        import torch.nn.functional as F
        w = self.window.to(img1.device)
        p = self.ws // 2
        mu1 = F.conv2d(img1, w, padding=p, groups=self.ch)
        mu2 = F.conv2d(img2, w, padding=p, groups=self.ch)
        mu1_sq,mu2_sq,mu1mu2 = mu1**2, mu2**2, mu1*mu2
        s1  = F.conv2d(img1*img1, w, padding=p, groups=self.ch) - mu1_sq
        s2  = F.conv2d(img2*img2, w, padding=p, groups=self.ch) - mu2_sq
        s12 = F.conv2d(img1*img2, w, padding=p, groups=self.ch) - mu1mu2
        C1, C2 = 0.01**2, 0.03**2
        ssim = ((2*mu1mu2+C1)*(2*s12+C2)) / ((mu1_sq+mu2_sq+C1)*(s1+s2+C2))
        return 1 - ssim.mean()

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=6, features=[64,128,256,512]):
        super().__init__()
        layers = []
        in_ch = in_channels
        for i, out_ch in enumerate(features):
            layers += [
                nn.Conv2d(in_ch, out_ch, 4, 1 if i==len(features)-1 else 2, 1, bias=False),
                nn.Identity() if i==0 else nn.BatchNorm2d(out_ch),
                nn.LeakyReLU(0.2, inplace=True)
            ]
            in_ch = out_ch
        layers.append(nn.Conv2d(in_ch, 1, 4, 1, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, input_img, target_img):
        return self.model(torch.cat([input_img, target_img], dim=1))

class EnhancementLoss(nn.Module):
    def __init__(self, lp=1.0, ls=1.0, ll=1.0):
        super().__init__()
        self.perc = PerceptualLoss()
        self.ssim = SSIMLoss()
        self.l1   = nn.L1Loss()
        self.bce  = nn.BCEWithLogitsLoss()

    def generator_loss(self, disc_fake, enhanced, target):
        # Frequency domain loss — recovers fine texture details
        freq_loss = torch.mean(torch.abs(
            torch.fft.fft2(enhanced) - torch.fft.fft2(target)
        ))
        adv  = self.bce(disc_fake, torch.ones_like(disc_fake))
        perc = self.perc(enhanced, target)
        ssim = self.ssim(enhanced, target)
        l1   = self.l1(enhanced, target)
        # KEY FIX: balanced weights — perceptual no longer dominates
        total = 0.05*adv + 0.25*perc + 0.50*ssim + 0.10*l1 + 0.10*freq_loss
        return total, {"adv": adv.item(), "perc": perc.item(),
                       "ssim": ssim.item(), "l1": l1.item(), "freq": freq_loss.item()}

    def discriminator_loss(self, real, fake):
        return (self.bce(real, torch.ones_like(real)) +
                self.bce(fake, torch.zeros_like(fake))) / 2
'''
with open('/kaggle/working/models/enhancement/discriminator.py', 'w') as f:
    f.write("from .losses import PatchGANDiscriminator\n")
with open('/kaggle/working/models/enhancement/losses.py', 'w') as f:
    f.write(loss_code)
with open('/kaggle/working/models/enhancement/__init__.py', 'w') as f:
    f.write('')
print("✅ Discriminator + losses written")


✅ Discriminator + losses written


In [6]:
import os

print("=== FILES ===")
files = [
    '/kaggle/working/models/__init__.py',
    '/kaggle/working/models/enhancement/__init__.py',
    '/kaggle/working/models/enhancement/generator.py',
    '/kaggle/working/models/enhancement/discriminator.py',
    '/kaggle/working/models/enhancement/losses.py',
]
for f in files:
    print(f"{'✅' if os.path.exists(f) else '❌'} {f}")

print("\n=== DATASET ===")
deg = len(os.listdir('/kaggle/working/data/degraded'))
cln = len(os.listdir('/kaggle/working/data/clean'))
print(f"✅ Degraded: {deg}")
print(f"✅ Clean:    {cln}")

print("\n=== CHECKPOINT ===")
ckpt = '/kaggle/input/datasets/ponnagaraj/checkpoint65/checkpoint_epoch_65.pt'
print(f"{'✅' if os.path.exists(ckpt) else '❌'} {ckpt}")

print(f"\n{'✅ READY TO TRAIN' if deg > 0 else '❌ NOT READY'}")


=== FILES ===
✅ /kaggle/working/models/__init__.py
✅ /kaggle/working/models/enhancement/__init__.py
✅ /kaggle/working/models/enhancement/generator.py
✅ /kaggle/working/models/enhancement/discriminator.py
✅ /kaggle/working/models/enhancement/losses.py

=== DATASET ===
✅ Degraded: 13791
✅ Clean:    13791

=== CHECKPOINT ===
✅ /kaggle/input/datasets/ponnagaraj/checkpoint65/checkpoint_epoch_65.pt

✅ READY TO TRAIN


In [ ]:
import sys, os, random
sys.path.insert(0, '/kaggle/working')

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

from models.enhancement.generator import UNetGenerator
from models.enhancement.discriminator import PatchGANDiscriminator
from models.enhancement.losses import EnhancementLoss

class PairedDataset(Dataset):
    def __init__(self, deg_dir, clean_dir, size=256):
        self.deg = Path(deg_dir)
        self.cln = Path(clean_dir)
        self.size = size
        exts = ['.jpg','.jpeg','.png']
        self.files = sorted([
            f.name for f in self.deg.iterdir()
            if f.suffix.lower() in exts
        ])
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
        ])
        print(f"Dataset: {len(self.files)} pairs")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        d = cv2.imread(str(self.deg/fname))
        c = cv2.imread(str(self.cln/fname))
        if d is None or c is None:
            z = torch.zeros(3, self.size, self.size)
            return z, z
        d = cv2.resize(cv2.cvtColor(d,cv2.COLOR_BGR2RGB),(self.size,self.size))
        c = cv2.resize(cv2.cvtColor(c,cv2.COLOR_BGR2RGB),(self.size,self.size))
        if np.random.random() > 0.5:
            d = np.fliplr(d).copy()
            c = np.fliplr(c).copy()
        if np.random.random() > 0.5:
            d = np.flipud(d).copy()
            c = np.flipud(c).copy()
        return self.tf(d), self.tf(c)

# CONFIG
EPOCHS      = 150
BATCH_SIZE  = 16
LR_G        = 0.0001
LR_D        = 0.00004
IMAGE_SIZE  = 256
SAVE_DIR    = '/kaggle/working/weights'
SAVE_EVERY  = 5
DEG_DIR     = '/kaggle/working/data/degraded'
CLEAN_DIR   = '/kaggle/working/data/clean'
RESUME_FROM = '/kaggle/input/datasets/ponnagaraj/checkpoint10/checkpoint_epoch_10.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")
os.makedirs(SAVE_DIR, exist_ok=True)

dataset    = PairedDataset(DEG_DIR, CLEAN_DIR, IMAGE_SIZE)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)

G = UNetGenerator().to(device)
D = PatchGANDiscriminator().to(device)
criterion = EnhancementLoss(lp=1.0, ls=1.0, ll=1.0).to(device)

opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5,0.999))
sch_G = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_G, T_0=25, T_mult=2)
sch_D = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_D, T_0=25, T_mult=2)

# Resume from epoch 10
start_epoch = 0
if os.path.exists(RESUME_FROM):
    print(f"Loading: {RESUME_FROM}")
    ckpt = torch.load(RESUME_FROM, map_location=device)
    G.load_state_dict(ckpt['generator'])
    D.load_state_dict(ckpt['discriminator'])
    if 'opt_G' in ckpt:
        opt_G.load_state_dict(ckpt['opt_G'])
        opt_D.load_state_dict(ckpt['opt_D'])
        print("Optimizer states restored ✅")
    start_epoch = ckpt['epoch'] + 1
    print(f"Resumed from epoch {start_epoch}")
else:
    print("Checkpoint not found — starting from scratch")

best_loss = float('inf')
g_losses, d_losses = [], []

for epoch in range(start_epoch, EPOCHS):
    G.train(); D.train()
    ep_g, ep_d = 0.0, 0.0

    for deg, clean in tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        deg, clean = deg.to(device), clean.to(device)

        with torch.inference_mode():
            fake = G(deg)
        d_loss = criterion.discriminator_loss(
            D(deg, clean), D(deg, fake.detach())
        )
        opt_D.zero_grad()
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        opt_D.step()

        fake = G(deg)
        g_loss, components = criterion.generator_loss(
            D(deg, fake), fake, clean
        )
        opt_G.zero_grad()
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G.step()

        ep_g += g_loss.item()
        ep_d += d_loss.item()

    avg_g = ep_g / len(dataloader)
    avg_d = ep_d / len(dataloader)
    sch_G.step()
    sch_D.step()
    g_losses.append(avg_g)
    d_losses.append(avg_d)

    print(f"Epoch {epoch+1}/{EPOCHS} | G: {avg_g:.4f} | D: {avg_d:.4f} | LR: {sch_G.get_last_lr()[0]:.6f}")

    if (epoch+1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'generator': G.state_dict(),
            'discriminator': D.state_dict(),
            'g_loss': avg_g,
            'opt_G': opt_G.state_dict(),
            'opt_D': opt_D.state_dict(),
        }, f'{SAVE_DIR}/checkpoint_epoch_{epoch+1}.pt')
        print(f"Checkpoint saved: epoch {epoch+1}")

    if avg_g < best_loss:
        best_loss = avg_g
        torch.save(G.state_dict(), f'{SAVE_DIR}/best_generator.pt')
        print(f"Best model saved — G_loss: {best_loss:.4f}")

    if (epoch+1) % 10 == 0:
        G.eval()
        with torch.inference_mode():
            sample_deg = next(iter(dataloader))[0][:4].to(device)
            sample_out = G(sample_deg)
            vutils.save_image(
                torch.cat([
                    sample_deg[:4]*0.5+0.5,
                    sample_out[:4]*0.5+0.5
                ], dim=0),
                f'{SAVE_DIR}/sample_epoch_{epoch+1}.png',
                nrow=4
            )
        G.train()
        print(f"Visual sample saved: sample_epoch_{epoch+1}.png")

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(g_losses, color='blue', label='G Loss')
plt.title('Generator Loss')
plt.xlabel('Epoch')
plt.legend()
plt.subplot(1,2,2)
plt.plot(d_losses, color='red', label='D Loss')
plt.title('Discriminator Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png')
plt.show()

print(f"\nTRAINING COMPLETE")
print(f"Best G Loss: {best_loss:.4f}")
print(f"Download: {SAVE_DIR}/best_generator.pt")


Training on: cuda
Dataset: 13790 pairs
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 230MB/s]  


Loading: /kaggle/input/datasets/ponnagaraj/checkpoint10/checkpoint_epoch_10.pt
Optimizer states restored ✅
Resumed from epoch 10


Epoch 11/150: 100%|██████████| 862/862 [13:15<00:00,  1.08it/s]


Epoch 11/150 | G: 1.6548 | D: 0.5318 | LR: 0.000100
Best model saved — G_loss: 1.6548


...

Epoch 28/150: 100%|██████████| 862/862 [13:16<00:00,  1.08it/s]


Epoch 28/150 | G: 1.6268 | D: 0.5209 | LR: 0.000018


Epoch 29/150: 100%|██████████| 862/862 [13:17<00:00,  1.08it/s]


Epoch 29/150 | G: 1.6228 | D: 0.5083 | LR: 0.000014
Best model saved — G_loss: 1.6228


Epoch 30/150:   6%|▌         | 49/862 [00:45<12:27,  1.09it/s]

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        if f.endswith('.pt'):
            print(os.path.join(dirname, f))


/kaggle/input/datasets/ponnagaraj/checkpoint65/checkpoint_epoch_65.pt
/kaggle/input/datasets/ponnagaraj/checkpoint10/checkpoint_epoch_10.pt


In [ ]:
import os

print("=== CHECKING SAVED FILES ===")
for f in sorted(os.listdir('/kaggle/working/weights')):
    size = os.path.getsize(f'/kaggle/working/weights/{f}') / 1e6
    print(f"{f} → {size:.1f} MB")
